In [17]:
import tensorflow as tf
import numpy as np

###Load MNIST Dataset

In [18]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

###Preprocessing

In [19]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train, x_test = x_train.reshape(-1, 784) / 255.0, x_test.reshape(-1, 784) / 255.0
x_train, x_test = x_train.astype(np.float32), x_test.astype(np.float32)


y_train, y_test = tf.one_hot(y_train, 10), tf.one_hot(y_test, 10)
y_train, y_test = tf.cast(y_train, tf.float32), tf.cast(y_test, tf.float32)

###Hyperparameters

In [20]:
learning_rate = 0.01
epochs = 10
batch_size = 256
lambda_l1 = 1e-5
lambda_l2 = 1e-6

###Model parameters

In [21]:
W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.1))
b1 = tf.Variable(tf.zeros([256]))

W2 = tf.Variable(tf.random.normal([256, 128], stddev=0.1))
b2 = tf.Variable(tf.zeros([128]))

W3 = tf.Variable(tf.random.normal([128, 10], stddev=0.1))
b3 = tf.Variable(tf.zeros([10]))

###Forward pass

In [22]:
def neural_net(x):
    layer1 = tf.nn.relu(tf.matmul(x, W1) + b1)
    layer2 = tf.nn.relu(tf.matmul(layer1, W2) + b2)
    logits = tf.matmul(layer2, W3) + b3
    return logits

###Loss function with L1 and L2 regularization

In [26]:
def compute_loss(x, y):
    logits = neural_net(x)
    cross_entropy = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(labels=y, logits=logits))


    l1_reg = lambda_l1 * (tf.reduce_sum(tf.abs(W1)) + tf.reduce_sum(tf.abs(W2)) + tf.reduce_sum(tf.abs(W3)))
    l2_reg = lambda_l2 * (tf.reduce_sum(tf.square(W1)) + tf.reduce_sum(tf.square(W2)) + tf.reduce_sum(tf.square(W3)))

    return cross_entropy + l1_reg + l2_reg

###Optimizer

In [23]:
optimizer = tf.optimizers.Adam(learning_rate)

#Training Loop

In [24]:
def train():
    for epoch in range(epochs):
        for i in range(0, len(x_train), batch_size):
            x_batch = x_train[i:i+batch_size]
            y_batch = y_train[i:i+batch_size]

            with tf.GradientTape() as tape:
                loss = compute_loss(x_batch, y_batch)

            grads = tape.gradient(loss, [W1, b1, W2, b2, W3, b3])
            optimizer.apply_gradients(zip(grads, [W1, b1, W2, b2, W3, b3]))


        logits = neural_net(x_test)
        correct_preds = tf.equal(tf.argmax(logits, 1), tf.argmax(y_test, 1))
        accuracy = tf.reduce_mean(tf.cast(correct_preds, tf.float32))

        print(f"Epoch {epoch+1}, Loss: {loss.numpy():.4f}, Accuracy: {accuracy.numpy():.4f}")


####train model

In [27]:
train()

Epoch 1, Loss: 0.2015, Accuracy: 0.9721
Epoch 2, Loss: 0.1899, Accuracy: 0.9705
Epoch 3, Loss: 0.2175, Accuracy: 0.9656
Epoch 4, Loss: 0.1574, Accuracy: 0.9718
Epoch 5, Loss: 0.1770, Accuracy: 0.9731
Epoch 6, Loss: 0.1764, Accuracy: 0.9729
Epoch 7, Loss: 0.1887, Accuracy: 0.9706
Epoch 8, Loss: 0.1553, Accuracy: 0.9694
Epoch 9, Loss: 0.1572, Accuracy: 0.9666
Epoch 10, Loss: 0.2147, Accuracy: 0.9678
